# Data extraction 
This notebook extracts all necessary data of the chosen X-Box for later plotting / processing. The data will be saved as pandas dataframes.

In [ ]:
import sys
sys.path.append('../') # Add parent directory to path (from env to parent)

import warnings
import scratch.processing_functions as pf
import scratch.post_processing_functions as ppf

import importlib
importlib.reload(pf)
import scratch.post_processing_functions_matt as ppfm

# User input

- `Xbox` is according to the chosen X-box (At the moment either 2 or 3).
- `structure_name` is according to the folder name on the DFS server (now cernbox folder).
- `file_name` is the appendix of the file the data will be saved to. This will be a costruction of the structure name and the file name.
- `startdate` and `enddate` are used to limit the extracted data. Only data after or at the startstate and before the enddate will be extracted.
- `plotting` needs to be True if the BD positions should be plotted.
- `t_fill` is the filling time of the structure used to calculate the BD time. With changing the strucutre, this can be adjusted.

From the `structure_name` the test stand (A, B or both) is known. If a new strucutre is used later, this needs to be adjusted.

The output data will be saved locally in the folder `processed_data` with the subfolder according to the structure name. If a different setup is wanted, `save_directory` needs to be adjusted.

In [5]:
# Select structure, i.e. both XBox number and structure (line)

structure_name='Xbox2_TD31N3N4'
structure_name="Xbox3_TD26CIEMAT_L1"

# Where to save processed tdms file (as h5)
folder_name='_pandas_new_test'

save_directory=(r"../data/"+structure_name+"/"+ folder_name + ".h5")
save_directory_trend=(r"../data/"+structure_name+"/"+folder_name+"_trend.h5")

# Date range to process
startdate=20260107
enddate=20260108

plotting = False

if structure_name=='Xbox2_TD31N3N4':
    structure_stand=0
    Xbox=2 
    t_fill=60e-9
    P_ref=36.1 #MW
    G_ref=72 #MV/m

elif structure_name=='Xbox3_TD26CIEMAT_L1':
    structure_stand=1
    Xbox=3
    t_fill=57.25e-9
    P_ref = 42.3  # [MW]
    G_ref = 100  # MV/m

## Getting all the data

This function is getting all the data for each group. 
It returns a Pandas dataframe which is then merged to the other data. The Dataframe includes:

- `pulse_count` the pulse count to identify the pulse
- `log_type` of the pulse
- `timestamp` of the pulse
- the PKI data: `PKI_total_power`,`PKI_length`,`PKI_peak`,`PKI_mean`,`PKI_start` (for Xbox-3 for Klystron A and B)
- the PKR data: `PKR_total_power`,`PKR_length`,`PKR_peak`,`PKR_mean`,`PKR_start` (for Xbox-3 for Klystron A and B)
- the compression gets checked bz the PKI length. If the PKI is at least 1e-6s, the pulse will be compressed Depending on this the `mid_pulse` as a reference is defined
- for XBox-3 the PLRA and PLRB data total power by the integral
- the PSI data: `PSIA_total_power`,`PSIA_peak`,`PSIA_peak_length`,`PSIA_mean_flat`,`PSIA_start`,`PSIA_t` for both or one structure
- the PSR data: `PSRA_total_power`,`PSRA_total_power_including_offset`,`PSRA_t`
- the PEI data: `PEIA_total_power`,`PEIA_peak`,`PEIA_peak_length`,`PEIA_mean_flat`,`PEIA_start`,`PEIA_t`
- the DC data: peak and integral of DC up and DC down
- there might be still an offset in the DC data, which should be removed

## Going though all groups in the TDMS file

- `load_directory` is the DFS server
- The drive can be mounted like:
    ```bash
    sudo mkdir /mnt/z/
    sudo mount -t drvfs '\\cernbox-drive\winspaces\x\xboxes' /mnt/z
    ```

In [6]:
# Directory of TDMS files (now on CERNBOX, not DFS)
# On windows and WSL, mount the network drive (Z:)
if structure_name=='Xbox2_TD31N3N4':
    # load_directory=r"//cernbox-drive/winspaces/x/xboxes/"+"Xbox2_TD31_N3N4"+"/"
    load_directory=r"/mnt/z/"+"Xbox2_TD31_N3N4"+"/"
else:
    load_directory=r"//cernbox-drive/winspaces/x/xboxes/"+structure_name+"/"  #NEW FOLDER (NOT DFS ANYMORE)
    #load_directory=r"/mnt/z/"+structure_name+"/"  #NEW FOLDER (NOT DFS ANYMORE)

#process all event data within chosen timeframe
pf.process_event_data(load_directory,save_directory,startdate,enddate,Xbox,structure_stand,plotting)

File:  EventDataA_20250707.tdms
File not used
File:  EventDataA_20250625.tdms_index
File not used
File:  EventDataA_20250724.tdms
File not used
File:  EventDataA_20250704.tdms_index
File not used
File:  EventDataA_20250626.tdms_index
File not used
File:  EventDataA_20251211.tdms_index
File not used
File:  EventDataA_20250712.tdms
File not used
File:  EventDataA_20250620.tdms_index
File not used
File:  EventDataA_20250612.tdms
File not used
File:  EventDataA_20251118.tdms
File not used
File:  EventDataA_20250716.tdms_index
File not used
File:  EventDataA_20251206.tdms
File not used
File:  EventDataA_20250706.tdms_index
File not used
File:  EventDataA_20250731.tdms
File not used
File:  EventDataA_20251210.tdms
File not used
File:  EventDataA_20251204.tdms
File not used
File:  EventDataA_20260109.tdms_index
File not used
File:  EventDataA_20250706.tdms
File not used
File:  EventDataA_20250924.tdms
File not used
File:  EventDataA_20250613.tdms_index
File not used
File:  EventDataA_20260112

# Update the File with all information
* Include the trend data
* Include the BDR caculation
* Include the BD pos calculation, t_fill is the filling time of the structure

In [12]:
load_directory=(r"../data/"+structure_name+"/"+structure_name+"_pandas_new_test.h5")
all_data=pf.load_event_data(load_directory)


In [15]:
#directory of DMS file
if structure_name=='Xbox2_TD31N3N4':
    #directory_trend=r"//cernbox-drive/winspaces/x/xboxes/"+"Xbox2_TD31_N3N4"+"/"
    directory_trend=r"/mnt/z/"+"Xbox2_TD31_N3N4"+"/"
else:
    #directory_trend=r"//cern.ch/dfs/Workspaces/x/"+structure_name+"/"
    #directory_trend=r"//cernbox-drive/winspaces/x/xboxes/"+structure_name+"/" #NEW FOLDER, not DFS ANYMORE
    directory_trend=r"/mnt/z/"+structure_name+"/"  #NEW FOLDER (NOT DFS ANYMORE)

#CLIC ref values
bdr_ref=1.7e-6 #bpp
pulse_length_ref= 180 #ns

all_data=ppf.adjust_variables(all_data,Xbox,structure_stand)
all_data=pf.add_gradient(all_data,P_ref,G_ref,Xbox,structure_stand)
all_data=pf.calculate_BDR(all_data,Xbox,structure_stand)
all_data=pf.calculate_BD_pos(all_data,Xbox,structure_stand,t_fill)
all_data=ppf.add_lost_power(all_data,Xbox,structure_stand)
all_data=ppfm.add_scaled_gradient(all_data,bdr_ref,pulse_length_ref,Xbox,structure_stand)

/home/lsito/XBox_Processing/notebooks/../src/processing_functions.py:712: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1e-06' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  input_data.loc[i, "BDR_HyPC_A"] = (input_data.loc[i, "cum_BD_HyPC_A"] - input_data.loc[last_pulse_j, "cum_BD_HyPC_A"]) / amount_BDR_sum
/home/lsito/XBox_Processing/notebooks/../src/processing_functions.py:710: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1e-06' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  input_data.loc[i, "BDR_DUT_A"] = ((input_data.loc[i, "cum_BD_DUT_withDC_A"] + input_data.loc[i, "cum_BD_DUT_withoutDC_A"]) - (input_data.loc[last_pulse_j, "cum_BD_DUT_withDC_A"] + input_data.loc[last_pulse_j, "cum_BD_DUT_withoutDC_A"])) / amount_BDR_sum
/home/lsito/

# Save the merged data 

In [16]:
#save new data
directory_save=(r"../data/"+structure_name+"/"+structure_name+"Structure_including_includingBD_new.h5")
#structure_data.to_hdf(directory_save, key=datetime.today().strftime('%Y-%m-%d'), mode='a')
all_data.to_hdf(directory_save, key=str(1), mode='a')

#save also in GUI folder
#directory_save=(r"GUI XBoxes/Data/"+structure_name+"Structure_including_trendData_includingBD_new.h5")
all_data.to_hdf(directory_save, key=str(1), mode='a')

/home/lsito/XBox_Processing/env/lib/python3.14/site-packages/tables/path.py:146: NaturalNameWarning: object name is not a valid Python identifier: '1'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/home/lsito/XBox_Processing/env/lib/python3.14/site-packages/tables/path.py:146: NaturalNameWarning: object name is not a valid Python identifier: '1'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
